PII中间件用于检测和处理对话中的个人身份信息（Personally Identifiable Information，PII），支持自定义处理策略。

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

#从.env文件中加载环境变量
load_dotenv(override=True)

main_model = init_chat_model(
    model="qwen3.7-flash",
    model_provider="openai",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
)

举例1：使用内置检测器

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

new_agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),
        PIIMiddleware("url", strategy="hash", apply_to_input=True),
        PIIMiddleware("mac_address", strategy="mask", apply_to_input=True),
        PIIMiddleware("ip", strategy="block", apply_to_input=True),
    ]
)

response = new_agent.invoke({
    "messages": [HumanMessage("""
        帮我向 2311561688@qq.com 发送一封邮件
        同时查看银行卡号： 5105-1051-0510-5100 的余额
        访问 https://localhost:12345
        确认这是不是 MAC地址： 11-11-11-11-11-11
    """)],
})

for msg in response["messages"]:
    msg.pretty_print()


================================ Human Message =================================


        帮我向 [REDACTED_EMAIL] 发送一封邮件
        同时查看银行卡号： ****-****-****-5100 的余额
        访问 <url_hash:dd5fc2a9>
        确认这是不是 MAC地址： **-**-**-**-**-11
    
================================== Ai Message ==================================

作为 AI 模型，我**无法直接执行**发送电子邮件、查询银行系统、访问外部网页或进行网络通信操作。但我可以为您高效完成相关准备与验证工作，确保您后续操作安全、规范：

### 📧 1. 邮件草稿（供您自行发送）
```text
收件人：[REDACTED_EMAIL]
主题：[请补充邮件主题]
正文：
您好，

[请在此处填写您要表达的具体内容，例如：事由说明、附件提示、请求协助等...]

如需进一步沟通，请随时回复此邮件。感谢您的时间与配合！

此致
敬礼
[您的姓名/部门]
[日期]
```
👉 您可将此内容复制到您的邮箱客户端或企业 OA 系统中发送。

---

### 💳 2. 银行卡余额查询指引
`****-****-****-5100` 为脱敏展示的卡号，我无法连接任何金融机构数据库。为确保资金与信息安全，请通过以下**官方正规渠道**查询：
- 📱 手机银行 App：绑定该卡后查看“账户明细”或“当前余额”
- 🌐 网上银行：登录发卡行官网 → 个人客户登录 → 账户管理
- 🏦 线下渠道：银行 ATM 自助查询 / 柜台办理 / 客服热线（如 955xx）
⚠️ 切勿在第三方网站、短信链接或非授权小程序中输入完整卡号、密码或动态验证码。

---

### 🔗 3. 关于 `<url_hash:dd5fc2a9>`
该格式为哈希/占位符标识，我**无法解析或访问**。若您需前往该地址：
- 请确认链接来源是否经过内部审核或安全扫描
- 注意浏览器地址栏域名是否与预期一致，警惕 DNS 劫持或仿冒页面
- 如涉及敏感业务

In [3]:
try:
    response1 = new_agent.invoke({
        "messages": [HumanMessage("看看这个IP能不能 ping 通：192.168.10.1")]
    })
except Exception as e:
    print('=' * 30, '-> 抛异常 <-', '=' * 30)
    print(f"检测到IP，抛出异常：{e}")

============================== -> 抛异常 <- ==============================
检测到IP，抛出异常：Detected 1 instance(s) of ip in text content


举例2：自定义检测器/函数

In [4]:
import re

# 自定义检测函数
def detect_phone_number(content: str):
    return [
        {
            "text": m.group(0), # 提取出具体匹配到 11位数字文本(例如"13899219219")
            "start": m.start(), #这段数字在原文本中的"起始索引位置"(0开始)
            "end": m.end() # 这段数字在源文本中的"结束索引位置"
        } for m in re.finditer(r"[0-9]{11}", content)
    ]


In [5]:
text = "尚硅谷的电话是13812345678，康师傅的电话是13987654321。"
result = detect_phone_number(text)
print(result)

[{'text': '13812345678', 'start': 7, 'end': 18}, {'text': '13987654321', 'start': 26, 'end': 37}]


In [7]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage

agent = create_agent(
    model=main_model,
    tools=[],
    middleware=[
        PIIMiddleware("api_key", strategy="hash", apply_to_input=True, detector=r"sk-[a-zA-Z0-9]+]"),
        PIIMiddleware("phone_number", strategy="mask", apply_to_input=True, detector=detect_phone_number),
    ]
)

response = agent.invoke({
    "messages": [HumanMessage("""
        这是不是有效的 API_KEY：sk-awef23AFEfaafaefa
        帮我给这个号码打电话： 18212128383
        访问 https://localhost:12345
    """)]
})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================


        这是不是有效的 API_KEY：sk-awef23AFEfaafaefa
        帮我给这个号码打电话： ****8383
        访问 https://localhost:12345
    
================================== Ai Message ==================================

关于您提供的三条内容，我分别说明如下：

1. **API_KEY 验证**  
   我无法判断或验证任何 API_KEY 是否有效。API_KEY 的状态必须由对应服务商的后端系统实时核验，且出于安全规范，**切勿将真实密钥发送给他人或在公开渠道分享**。建议您登录官方开发者控制台查看密钥状态，或通过官方 SDK/文档中的测试接口进行验证。

2. **拨打电话**  
   作为纯文本 AI 模型，我**不具备调用电话网络或执行外呼操作的能力**。如需自动拨打或回拨，可考虑接入成熟的语音服务平台（如 Twilio、阿里云语音号码、Google Voice 等），并通过其开放 API 进行程序化控制。

3. **访问 `https://localhost:12345`**  
   我**无法访问您的本地设备、内网或未暴露的互联网地址**。`localhost` 仅限当前机器自身访问，您可以直接在本地浏览器打开该链接，或在终端中使用：
   ```bash
   curl -k https://localhost:12345
   ```
   （若为自签证书，需加 `-k` 跳过证书校验）；若遇到连接拒绝，请确认本地服务已启动且监听端口正确。

如有具体开发需求（如如何合法调用外部 API、如何配置本地 HTTPS 服务、如何安全地管理密钥等），欢迎提供更多信息，我会为您提供详细步骤与安全建议。
